In [ ]:
import pandas as pd

df = pd.read_csv('2000_raws.csv', encoding='utf-8')

new_df = df[1500:1750]
new_df.to_csv('texts_for_classifier.csv', index=False)

In [ ]:
import pandas as pd
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "/content/sft_model"
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_path).to(device)
model.eval()

In [ ]:
df = pd.read_csv('texts_for_classifier.csv', encoding='utf8')
texts = df['Заголовок'].dropna().tolist()

In [ ]:
def generate_headlines(model, input_texts, num_headlines=1, num_beams=4, max_new_tokens=50):
    headlines = []
    for text in input_texts:
        prompt = text + tokenizer.eos_token
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=num_beams,
                repetition_penalty=1.15,
                no_repeat_ngram_size=3,
                early_stopping=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
        parts = decoded.split(tokenizer.eos_token)
        if len(parts) >= 2:
            paraphrase = parts[1].strip()
        else:
            paraphrase = decoded.strip()
        paraphrase = paraphrase.replace(tokenizer.eos_token, '').strip()
        headlines.append(paraphrase)
    return headlines

# Генерация перефразирований
paraphrases = []
for text in tqdm(texts, desc="Генерация перефразирваний SFT моделью"):
    para = generate_headlines(text)
    paraphrases.append(para)

result_df = pd.DataFrame({
    'исходный_заголовок': texts,
    'перефразирование': paraphrases
})
result_df.to_csv('sft_paraphrases.csv', index=False, encoding='utf-8-sig', na_rep='')